<a href="https://colab.research.google.com/github/rduarte12/jet_tagging_gnn/blob/main/notebooks/jet_tagging_gnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# install dependences
!pip install zenodo_get torch_geometric h5py -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 20.8 MB/s eta 0:00:00


In [3]:
# mount Google Drive to ensure data persistence.
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/jet_tagging'

Mounted at /content/drive


In [ ]:
# download dataset
!zenodo_get 2603256 -o /content/data/

In [ ]:
import h5py

def explore_hdf5(f, indent=0):
    for key in f.keys():
        item = f[key]
        prefix = "  " * indent
        if isinstance(item, h5py.Group):
            print(f"{prefix}Group: {key}/")
            explore_hdf5(item, indent + 1)
        elif isinstance(item, h5py.Dataset):
            print(f"{prefix}Dataset: {key} | shape={item.shape} | dtype={item.dtype}")

with h5py.File('/content/data/train.h5', 'r') as f:
    explore_hdf5(f)

In [ ]:
import pandas as pd

# Load only 5 rows to inspect
df = pd.read_hdf('/content/data/train.h5', key='table', stop=5)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Dtypes:\n", df.dtypes.value_counts())

In [ ]:
# check how are the features, the labels and verify some important variables
import h5py
import numpy as np

with h5py.File('/content/data/train.h5', 'r') as f:
    print("Keys:", list(f.keys()))
    X_sample = f['<key_features>'][:5]
    y_sample = f['<key_labels>'][:5]

# Shape and labels
print("Feature shape:", X_sample.shape)
print("Labels:", y_sample)

# pT sanity check
print("\n--- pT ---")
print("pT min/max:", X_sample[:,:,0].min(), X_sample[:,:,0].max())

# Real particles per jet (non-padding)
n_real = (X_sample[:,:,0] > 0).sum(axis=1)
print("\n--- Real particles per jet (pT > 0) ---")
print("Count per jet:", n_real)
print("Min/max:", n_real.min(), n_real.max())
print("Mean:", n_real.mean())

# Feature ranges (real particles only)
mask = X_sample[:,:,0] > 0
print("\n--- Feature ranges (real particles only) ---")
print("eta range:", X_sample[:,:,1][mask].min(), X_sample[:,:,1][mask].max())
print("phi range:", X_sample[:,:,2][mask].min(), X_sample[:,:,2][mask].max())
print("E range:  ", X_sample[:,:,3][mask].min(), X_sample[:,:,3][mask].max())

# Label balance
print("\n--- Label balance ---")
print("Signal fraction:", y_sample.mean())

# Padding inspection
padding_mask = X_sample[:,:,0] == 0
padding_vectors = X_sample[padding_mask]
print("\n--- Padding vectors ---")
print("Padding count per jet:", padding_mask.sum(axis=1))
print("All padding are zeros:", (padding_vectors == 0).all())
if not (padding_vectors == 0).all():
    print("Non-zero values in padding:", padding_vectors[padding_vectors != 0])